| Feature           | Comes from         | Calculation                |
| ----------------- | ------------------ | -------------------------- |
| HAG               | Ground surface     | Z - ground elevation       |
| Normal vector     | Neighboring points | PCA on neighborhood        |
| Planarity         | Eigenvalues        | PCA                        |
| Linearity         | Eigenvalues        | PCA                        |
| Sphericity        | Eigenvalues        | PCA                        |
| Surface variation | Eigenvalues        | PCA                        |
| Omnivariance      | Eigenvalues        | PCA                        |
| Eigenentropy      | Eigenvalues        | PCA                        |
| Roughness         | Neighboring points | Distance to best-fit plane |
| Density           | Neighbor search    | Points within radius       |
| Height statistics | Neighboring points | min/max/std of Z           |


In [1]:
#!/usr/bin/env python

############################################################
# Use Numpy to Calculate LAS variables
# Ian Horn
# August 7, 2026
############################################################

import json 
import pdal
import time
import numpy as np
from scipy.spatial import cKDTree


In [2]:
lasfile = "/mnt/d/Data/lidar/N075E299.laz"

pipeline = {
    "pipeline": [
        lasfile,
        {"type": "filters.hag_delaunay"}
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))

count = p.execute()

arrays = p.arrays

points = arrays[0]

# print(points.dtype)
# print(points.dtype.names)
# print(points.shape)

# Height Above Ground (HAG)

Added `filters.hag_delaunay` to the PDAL pipeline — it builds a ground TIN from ground-classified points (Classification == 2, ~8.6M of 12.3M points here) and computes each point's height above that surface as a new `HeightAboveGround` dimension. Runs in PDAL/C++, ~20s on the full cloud.

`HeightAboveGround` comes out in feet, same as everything else here, since it's just `Z - interpolated_ground_Z`.

Min/max come out at roughly -46ft / +266ft — those extremes are almost entirely noise-classified points (Classification 7/18, ~5.1k points total), not real geometry. Worth masking those classes out before using HAG downstream (e.g. for building height thresholds).

In [3]:
hag = points["HeightAboveGround"]

noise_mask = np.isin(points["Classification"], [7, 18])
print(f"HAG stats (all points): min={hag.min():.2f} max={hag.max():.2f} mean={hag.mean():.2f}")
print(f"HAG stats (excl. noise classes 7/18, n={noise_mask.sum():,}): "
      f"min={hag[~noise_mask].min():.2f} max={hag[~noise_mask].max():.2f} mean={hag[~noise_mask].mean():.2f}")
print(f"fraction near ground (|HAG| < 0.5ft): {(np.abs(hag) < 0.5).mean():.3f}")

HAG stats (all points): min=-45.67 max=266.27 mean=7.33
HAG stats (excl. noise classes 7/18, n=5,140): min=-18.83 max=150.86 mean=7.31
fraction near ground (|HAG| < 0.5ft): 0.722


In [4]:
xyz = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"]
])

# xyz


In [5]:
features = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"],
    points["Intensity"],
    points["Classification"]
])

# features


In [6]:
xyz = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"]
])

tree = cKDTree(xyz)
# tree

print(len(xyz), "points loaded into array")


12294882 points loaded into array


In [7]:
# # example of finding neighbors within a radius of 3.0 units from the second point in the dataset
# point = xyz[1]
# idx = tree.query_ball_point(point, r=3.0)
# neighbors = xyz[idx]
# neighbors


# Compute Geometric Features

# Vectorized version

The loop above calls `query_ball_point`, `np.cov`, and `np.linalg.eigh` once per point. At 12.3M points that's slow (~11 min extrapolated).

This version:
- batches the radius search across all points in one `query_ball_point` call (parallelized via `workers=-1`)
- accumulates each point's covariance matrix with `np.bincount` segment sums over the flattened (point, neighbor) pairs, instead of a per-point `np.cov`
- eigendecomposes the whole stack of 3x3 covariance matrices in one `np.linalg.eigh` call per chunk

Processed in chunks (default 300k points) because the full flattened neighbor-pair list (~130M pairs for this cloud at radius=3.0) doesn't comfortably fit in memory at once.

Verified against the loop version on a 2000-point random sample: max abs difference ~1e-15 (floating point noise). On the full 12.3M point cloud this ran in ~36s vs. ~11 min extrapolated for the loop version (~19x speedup).

Also returns `NeighborCount` and `Density` (points per cubic foot, `count / ((4/3)*pi*radius**3)`) — free, since the radius search for the PCA features already produces the neighbor count per point. Density isn't gated by `min_neighbors` like the PCA features are; it's meaningful even for sparse points.

`Verticality = 1 - |NormalZ|` — 0 for a horizontal surface (normal points straight up/down), 1 for a vertical surface (normal perpendicular to Z). Same definition PDAL's `filters.covariancefeatures` uses. Also free — just a function of the normal that's already computed.

In [ ]:
def compute_geometric_features_vectorized(tree, xyz, radius, min_neighbors=5, chunk_size=300_000):

    n_points = len(xyz)
    sphere_volume = (4.0 / 3.0) * np.pi * radius ** 3

    out = {
        "Planarity": np.zeros(n_points),
        "Linearity": np.zeros(n_points),
        "Sphericity": np.zeros(n_points),
        "SurfaceVariation": np.zeros(n_points),
        "Roughness": np.zeros(n_points),
        "Verticality": np.zeros(n_points),
        "NormalX": np.zeros(n_points),
        "NormalY": np.zeros(n_points),
        "NormalZ": np.zeros(n_points),
        "NeighborCount": np.zeros(n_points, dtype=np.int64),
        "Density": np.zeros(n_points),
    }

    for start in range(0, n_points, chunk_size):
        stop = min(start + chunk_size, n_points)
        chunk_idx = np.arange(start, stop)
        chunk_xyz = xyz[chunk_idx]
        n_chunk = len(chunk_idx)

        # Batched, parallel radius search for the whole chunk
        neighbor_lists = tree.query_ball_point(chunk_xyz, r=radius, workers=-1)
        lengths = np.fromiter((len(nb) for nb in neighbor_lists), dtype=np.int64, count=n_chunk)
        valid = lengths >= min_neighbors

        # Density is defined for every point regardless of the PCA min_neighbors floor --
        # it's just neighbor count over the search-sphere volume, points per cubic foot
        out["NeighborCount"][chunk_idx] = lengths
        out["Density"][chunk_idx] = lengths / sphere_volume

        # Flatten (point, neighbor) pairs
        point_ids = np.repeat(np.arange(n_chunk), lengths)
        neighbor_ids = np.concatenate(neighbor_lists) if lengths.sum() else np.empty(0, dtype=np.int64)
        neighbor_xyz = xyz[neighbor_ids]

        # Centroid per point via segment sum
        safe_lengths = np.maximum(lengths, 1)
        sums = np.zeros((n_chunk, 3))
        np.add.at(sums, point_ids, neighbor_xyz)
        centroid = sums / safe_lengths[:, None]

        centered = neighbor_xyz - centroid[point_ids]

        # Covariance matrix per point, accumulated with bincount instead of per-point np.cov
        cxx = np.bincount(point_ids, weights=centered[:, 0] * centered[:, 0], minlength=n_chunk)
        cxy = np.bincount(point_ids, weights=centered[:, 0] * centered[:, 1], minlength=n_chunk)
        cxz = np.bincount(point_ids, weights=centered[:, 0] * centered[:, 2], minlength=n_chunk)
        cyy = np.bincount(point_ids, weights=centered[:, 1] * centered[:, 1], minlength=n_chunk)
        cyz = np.bincount(point_ids, weights=centered[:, 1] * centered[:, 2], minlength=n_chunk)
        czz = np.bincount(point_ids, weights=centered[:, 2] * centered[:, 2], minlength=n_chunk)

        denom = np.maximum(lengths - 1, 1)  # ddof=1, matches np.cov default
        cov = np.zeros((n_chunk, 3, 3))
        cov[:, 0, 0] = cxx / denom
        cov[:, 1, 1] = cyy / denom
        cov[:, 2, 2] = czz / denom
        cov[:, 0, 1] = cov[:, 1, 0] = cxy / denom
        cov[:, 0, 2] = cov[:, 2, 0] = cxz / denom
        cov[:, 1, 2] = cov[:, 2, 1] = cyz / denom

        # Batched eigendecomposition of the whole chunk's covariance matrices at once.
        # eigh returns ascending eigenvalues: index 0 = smallest (l3), index 2 = largest (l1)
        eigvals, eigvecs = np.linalg.eigh(cov)
        l3, l2, l1 = eigvals[:, 0], eigvals[:, 1], eigvals[:, 2]
        normal = eigvecs[:, :, 0]

        valid = valid & (l1 > 0)

        with np.errstate(divide="ignore", invalid="ignore"):
            linearity = np.where(valid, (l1 - l2) / l1, 0.0)
            planarity = np.where(valid, (l2 - l3) / l1, 0.0)
            sphericity = np.where(valid, l3 / l1, 0.0)
            total = l1 + l2 + l3
            surface_variation = np.where(valid & (total > 0), l3 / total, 0.0)

        # Roughness = std of distance to best-fit plane, via segment sum/sum-of-squares
        dist = np.einsum("ij,ij->i", centered, normal[point_ids])
        sum_d = np.bincount(point_ids, weights=dist, minlength=n_chunk)
        sum_d2 = np.bincount(point_ids, weights=dist ** 2, minlength=n_chunk)
        mean_d = sum_d / safe_lengths
        roughness = np.sqrt(np.maximum(sum_d2 / safe_lengths - mean_d ** 2, 0.0))
        roughness = np.where(valid, roughness, 0.0)

        # Verticality: 0 for a horizontal surface (normal ~ +-Z), 1 for a vertical
        # surface (normal perpendicular to Z) -- same definition PDAL's
        # filters.covariancefeatures uses.
        verticality = np.where(valid, 1.0 - np.abs(normal[:, 2]), 0.0)

        normal = np.where(valid[:, None], normal, 0.0)

        out["Planarity"][chunk_idx] = planarity
        out["Linearity"][chunk_idx] = linearity
        out["Sphericity"][chunk_idx] = sphericity
        out["SurfaceVariation"][chunk_idx] = surface_variation
        out["Roughness"][chunk_idx] = roughness
        out["Verticality"][chunk_idx] = verticality
        out["NormalX"][chunk_idx] = normal[:, 0]
        out["NormalY"][chunk_idx] = normal[:, 1]
        out["NormalZ"][chunk_idx] = normal[:, 2]

    return out

In [9]:
start = time.perf_counter()
geom_vec = compute_geometric_features_vectorized(tree, xyz, radius=3.0)
elapsed = time.perf_counter() - start

print(f"Elapsed: {elapsed:.1f} seconds")
# geom_vec

Elapsed: 42.8 seconds


# Radius = 7 ft

The point cloud CRS (`NAD83 / Kentucky Single Zone ftUS`) means radius is in feet, not meters. Swept radius=4/5/7/10 ft and compared coverage, neighbor stability, and feature distributions:

| radius (ft) | coverage % | mean neighbors |
|---|---|---|
| 4 | 94.2 | 17.4 |
| 5 | 97.2 | 27.8 |
| 7 | 99.5 | 55.5 |
| 10 | 100.0 | 116.5 |

Neither Planarity nor Linearity plateaus across this range, so there's no clean converged optimum. Chose **7ft**: coverage reaches 99.5% with stable neighbor counts, without going as far as 10ft where Roughness p90 balloons to 2.7ft — a sign the neighborhood is starting to pick up whole-building-scale structure rather than local surface texture.

In [10]:
start = time.perf_counter()
geom_vec = compute_geometric_features_vectorized(tree, xyz, radius=7.0, chunk_size=90_000)
elapsed = time.perf_counter() - start

print(f"Elapsed: {elapsed:.1f} seconds")

Elapsed: 163.7 seconds
